# Hugging Face in one pattern: tokeniser, model, outputs

**Block 1, worked example.** The lecture used `SentenceTransformer(...)` and `model.tokenizer` without explaining the libraries behind them. This notebook shows the basic syntax of the Hugging Face libraries that almost every open model is used with: the **Hub** (`huggingface_hub`), where models are published and downloaded; **transformers**, with one pattern that works for every model and task; the **`pipeline()`** shortcut; and **sentence-transformers**, the library of the course's embedding model. Once you know the pattern, a new model is usually a change of one name.

Uses `transformers`, `sentence-transformers` and `huggingface_hub` from the course environment (developed with transformers 5.19, sentence-transformers 6.1 and huggingface_hub 1.33). Everything runs on the CPU. Besides `intfloat/multilingual-e5-small` (about 470 MB, used throughout Session 14), the first run downloads three small models for the `pipeline()` section: about 270 MB, 560 MB and 440 MB. No API key is needed.

Theory: [01-lecture-embeddings-from-words-to-vectors.ipynb](01-lecture-embeddings-from-words-to-vectors.ipynb).

```mermaid
flowchart LR
    T["Texts"] --> K["Tokeniser<br/>AutoTokenizer"]
    K --> I["input_ids +<br/>attention_mask<br/>(tensors)"]
    I --> M["Model<br/>AutoModel..."]
    M --> O["Outputs<br/>last_hidden_state<br/>or logits"]
    O --> P["Post-processing<br/>pooling, softmax,<br/>id2label"]
    P --> R["Vectors, labels,<br/>scores"]
```

*Author: course team, licence CC BY 4.0.*

In [1]:
import huggingface_hub
import sentence_transformers
import torch
import transformers

print("transformers", transformers.__version__, "| sentence-transformers", sentence_transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| torch", torch.__version__)

.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.19.0 | sentence-transformers 6.1.0 | huggingface_hub 1.33.0 | torch 2.14.1


## 1. The Hub: a model is a name, a model card and a licence

Every model on the [Hugging Face Hub](https://huggingface.co/models) is identified by its name `organisation/model`, for example `intfloat/multilingual-e5-small`. The model's page is its **model card**: what it was trained on, what it is for, its limits and its **licence**. Read it before you use a model; the licence decides whether you may use it in a project (Apache-2.0 and MIT allow commercial use, "NC" licences do not).

`from_pretrained(name)` downloads the model files once into the local **cache** (by default `~/.cache/huggingface/hub`; the environment variable `HF_HOME` moves it) and loads them from there afterwards, also offline. A model repository changes over time; `revision=` pins a version (a branch, a tag or a commit hash), so a result can be reproduced later.

In [2]:
from huggingface_hub import constants, model_info

name = "intfloat/multilingual-e5-small"
info = model_info(name)                      # metadata from the Hub (needs internet)
print(info.id, "| task:", info.pipeline_tag, "| licence:", info.card_data.get("license"))
print("current commit:", info.sha)
print("cache folder:", constants.HF_HUB_CACHE)

intfloat/multilingual-e5-small | task: sentence-similarity | licence: mit
current commit: 614241f622f53c4eeff9890bdc4f31cfecc418b3
cache folder: ~/.cache/huggingface/hub


> **Tip.** Pin the commit in code that must give the same result next year: `AutoModel.from_pretrained(name, revision="614241f...")` with the full hash printed above. Without `revision`, you get whatever the repository's main branch holds on the day of the first download.

## 2. The one pattern of transformers: tokeniser, model, outputs

Every model in `transformers` is used in the same three steps.

1. **Tokeniser.** `AutoTokenizer.from_pretrained(name)` loads the tokeniser that belongs to the model. Calling it on a list of texts returns a dictionary of tensors: `input_ids` (the token ids) and `attention_mask` (1 for real tokens, 0 for padding). `padding=True` pads the shorter texts to the length of the longest, `truncation=True` cuts texts at the model's input limit, `return_tensors="pt"` returns PyTorch tensors.
2. **Model.** `AutoModel.from_pretrained(name)` reads the model's configuration and builds the right architecture (here a `BertModel`). The `Auto` classes save you from knowing the class name.
3. **Outputs.** `model(**inputs)` passes the dictionary as keyword arguments. Wrap it in `torch.no_grad()`: we only predict, so PyTorch need not store what it would need for training. The result is an object with named fields; an encoder returns `last_hidden_state`, one vector per token.

In [3]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(name)
texts = ["query: Damenschuhe mit Oberteil aus Leder",
         "query: Voiture jouet en matière plastique, avec roues"]
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
print(list(inputs.keys()), inputs["input_ids"].shape)          # 2 texts, padded to the same length
print(inputs["input_ids"][0])
print(inputs["attention_mask"])                                 # 0 = padding, ignored by the model
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))  # <s> and </s> mark start and end

['input_ids', 'attention_mask'] torch.Size([2, 16])
tensor([     0,     41,   1294,     12, 170936, 230233,    491,  32629,  35617,
          1312, 118281,      2,      1,      1,      1,      1])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
['<s>', '▁que', 'ry', ':', '▁Damen', 'schuhe', '▁mit', '▁Ober', 'teil', '▁aus', '▁Leder', '</s>', '<pad>', '<pad>', '<pad>', '<pad>']


In [4]:
model = AutoModel.from_pretrained(name)
print(type(model).__name__, "| training mode:", model.training)   # from_pretrained returns the model in evaluation mode

with torch.no_grad():
    outputs = model(**inputs)
print(type(outputs).__name__)
print(outputs.last_hidden_state.shape)       # (texts, tokens, 384): one 384-dimensional vector per token

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26231.70it/s]

BertModel | training mode: False
BaseModelOutputWithPoolingAndCrossAttentions
torch.Size([2, 16, 384])


Post-processing turns the outputs into what we need. For a sentence embedding, average the token vectors of the real tokens (**mean pooling**, using the attention mask) and scale the result to length 1. This is exactly what sentence-transformers does for this model (section 4).

In [5]:
mask = inputs["attention_mask"].unsqueeze(-1)                       # (texts, tokens, 1)
pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
manual = torch.nn.functional.normalize(pooled, dim=-1)              # length 1
print(manual.shape, round(float(manual[0] @ manual[1]), 4))         # cosine similarity of the two texts

torch.Size([2, 384]) 0.7745


**The same pattern for another task.** A model trained for classification has a classification layer on top; it is loaded with `AutoModelForSequenceClassification` and returns **logits**, one score per class. Softmax turns them into probabilities, and the model's configuration maps class numbers to names (`id2label`). The example model was fine-tuned on English movie reviews (SST-2), so we give it English product reviews.

In [6]:
from transformers import AutoModelForSequenceClassification

clf_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
clf_tokenizer = AutoTokenizer.from_pretrained(clf_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_name)

reviews = ["The shoes arrived broken.", "Great quality toy car, my son loves it."]
batch = clf_tokenizer(reviews, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    logits = clf_model(**batch).logits              # (texts, classes)
probs = logits.softmax(dim=-1)
print(logits.round(decimals=2))
print(probs.round(decimals=4))
print(clf_model.config.id2label)
print([clf_model.config.id2label[int(i)] for i in probs.argmax(dim=-1)])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 24684.94it/s]

tensor([[ 4.1500, -3.3800],
        [-4.2800,  4.5700]])
tensor([[9.9950e-01, 5.0000e-04],
        [1.0000e-04, 9.9990e-01]])
{0: 'NEGATIVE', 1: 'POSITIVE'}
['NEGATIVE', 'POSITIVE']


> 📖 **Read more:** [extras/01-hf-course-tokenizers.ipynb](extras/01-hf-course-tokenizers.ipynb) (Hugging Face course): the tokeniser step in detail: tokens, ids, decoding, special tokens and saving a tokeniser.

## 3. The shortcut: `pipeline()`

`pipeline(task, model=name)` bundles the three steps and the post-processing for a task. The call shape is the same for every task: create the pipeline once, then call it on a text or a list of texts. Some common task names: `"sentiment-analysis"` (text classification), `"zero-shot-classification"`, `"fill-mask"`, `"feature-extraction"`, `"translation"`, `"text-generation"`. Always pass `model=` explicitly: without it, `pipeline()` picks a default model that can change between library versions.

In [7]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model=clf_name, device="cpu")
print(sentiment(reviews))                          # the same model as above, in one line

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 25730.41it/s]

[{'label': 'NEGATIVE', 'score': 0.9994667172431946}, {'label': 'POSITIVE', 'score': 0.9998573064804077}]


**Zero-shot classification** needs no training data for the labels: a model trained on natural-language inference checks, for each candidate label, whether the text supports the sentence "This product is {label}." The multilingual model below handles descriptions in German and French with English labels.

In [8]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device="cpu")
descriptions = ["Damenstiefel mit Oberteil aus Rindleder und Laufsohle aus Gummi",
                "Voiture jouet en matière plastique, avec roues",
                "Kabellose Laptop-Maus mit USB-Empfänger",
                "Pull-over en maille de coton pour hommes"]
labels = ["footwear", "toys", "computer equipment", "clothing"]
for r in zero_shot(descriptions, candidate_labels=labels, hypothesis_template="This product is {}."):
    print(f"{r['labels'][0]:20s} {r['scores'][0]:.2f}  {r['sequence']}")

.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 24325.28it/s]

footwear             0.92  Damenstiefel mit Oberteil aus Rindleder und Laufsohle aus Gummi
toys                 0.94  Voiture jouet en matière plastique, avec roues
computer equipment   0.90  Kabellose Laptop-Maus mit USB-Empfänger
clothing             0.99  Pull-over en maille de coton pour hommes


**Fill-mask** shows what an encoder learned in pre-training (masked language modelling, lecture): it predicts the hidden token. The German BERT model uses `[MASK]` as its mask token; other models use other strings, which `fill_mask.tokenizer.mask_token` tells you. The load report about "UNEXPECTED" keys is harmless: the downloaded checkpoint also contains layers (the pooler and the next-sentence-prediction head of BERT pre-training) that the fill-mask model does not use.

In [9]:
fill_mask = pipeline("fill-mask", model="google-bert/bert-base-german-cased", device="cpu")
print(fill_mask.tokenizer.mask_token)
for r in fill_mask("Schuhe mit Oberteil aus [MASK] und Laufsohle aus Gummi.", top_k=3):
    print(f"{r['token_str']:12s} {r['score']:.2f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 23288.24it/s]


[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-german-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MASK]
Leder        0.36
Kunststoff   0.13
Stoff        0.11


> **Caution.** A pipeline hides the tokeniser and the model, but not their limits: the sentiment model only knows English, the zero-shot model only knows the labels you give it and is slow with many labels (one model call per label and text), and every model cuts long texts at its input limit. Check a new pipeline on labelled examples before you trust it.

> 📖 **Read more:** [extras/03-hf-course-transformer-pipelines.ipynb](extras/03-hf-course-transformer-pipelines.ipynb) (Hugging Face course): `pipeline()` for many more tasks, including text generation, named-entity recognition, summarisation and translation (it downloads several models, some larger than 1 GB).

## 4. sentence-transformers: one call for embeddings

For embeddings, `sentence-transformers` wraps the pattern of section 2 (tokeniser, model, pooling, normalisation) into `encode`. `model.similarity` (or `sentence_transformers.util.cos_sim`) computes all pairwise cosine similarities. The result equals our manual mean pooling above.

In [10]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer(name, device="cpu")
E = st_model.encode(texts, normalize_embeddings=True, convert_to_tensor=True)
print(E.shape)
print(st_model.similarity(E, E))                     # cosine similarity matrix
print(util.cos_sim(E, E))                            # the same with the utility function
print("equal to the manual version:", torch.allclose(E, manual, atol=1e-5))
print(st_model.tokenizer is not None, st_model.max_seq_length)   # the tokeniser of section 2 is inside

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25961.63it/s]

torch.Size([2, 384])
tensor([[1.0000, 0.7745],
        [0.7745, 1.0000]])
tensor([[1.0000, 0.7745],
        [0.7745, 1.0000]])
equal to the manual version: True
True 512


## Cheat sheet

```python
# the Hub
from huggingface_hub import model_info
model_info("org/model").card_data.get("license")             # licence from the model card

# the pattern (any model, any task)
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("org/model", revision="<commit>")    # revision pins a version
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")  # input_ids, attention_mask
model = AutoModel.from_pretrained("org/model")              # or AutoModelFor<Task>
with torch.no_grad():
    outputs = model(**inputs)                               # .last_hidden_state or .logits
probs = outputs.logits.softmax(dim=-1)                      # post-processing for classifiers
model.config.id2label                                       # class numbers -> names

# the shortcut
from transformers import pipeline
pipe = pipeline("zero-shot-classification", model="org/model", device="cpu")   # always name the model
pipe(texts, candidate_labels=["a", "b"])

# embeddings
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
E = st.encode(["query: ..."], normalize_embeddings=True)
st.similarity(E, E)
```

## Check your understanding

1. What do `input_ids` and `attention_mask` contain for a batch of two texts of different length? What would go wrong in mean pooling without the attention mask?
2. You load `AutoModel` and `AutoModelForSequenceClassification` from the same name. What is the difference in the outputs?
3. Why should you always pass `model=` to `pipeline()`, and why `revision=` in code you hand in?
4. A model card says "license: cc-by-nc-4.0". May your team use the model in a product for a customs broker?
5. Replace the name `intfloat/multilingual-e5-small` by `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` in section 4. Which other line must change, and why? (Hint: the prefixes.)

## Further reading

- Hugging Face. *LLM Course*, chapter 1 (transformer models and `pipeline()`) and chapter 2 (using transformers: tokenisers, models, the pattern of this notebook). https://huggingface.co/learn/llm-course/chapter1/1 and https://huggingface.co/learn/llm-course/chapter2/1
- Hugging Face. *Transformers: quick tour*. https://huggingface.co/docs/transformers/quicktour
- Sentence-Transformers. *Quickstart*. https://sbert.net/docs/quickstart.html
- Hugging Face. *Model cards* and *Manage the cache*. https://huggingface.co/docs/hub/model-cards · https://huggingface.co/docs/huggingface_hub/guides/manage-cache